# ⚡ Probando Groq Cloud Directo y con LangChain

Este notebook forma parte del curso **Introducción a la Inteligencia Artificial Generativa**.

**Groq** es una plataforma de inferencia ultra-rápida basada en hardware especializado llamado **LPU** (Language Processing Units). Permite ejecutar modelos de código abierto (como LLaMA 3 y Mixtral) con velocidades de cientos de tokens por segundo.

En este notebook aprenderás a:
1. Usar el **SDK oficial de Groq (`groq`)** directamente.
2. Medir la latencia y tokens por segundo.
3. Integrar Groq con **LangChain (`langchain-groq`)** y construir cadenas con LCEL.


## 📦 1. Instalación de dependencias

Si ejecutas este notebook por primera vez en un entorno nuevo, descomenta y ejecuta la siguiente celda:


In [ ]:
# Descomenta la siguiente línea para instalar dependencias si es necesario:
# !pip install -q python-dotenv groq langchain-groq langchain-core


## 🔑 2. Carga y verificación de la API Key

Usamos `python-dotenv` para cargar la variable `GROQ_API_KEY` desde el archivo `.env` en la raíz del repositorio.

> **Nota:** Puedes obtener una clave de API gratuita en la [Consola de Groq](https://console.groq.com/keys).


In [4]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde el archivo .env en la raíz del repositorio
load_dotenv(dotenv_path="../.env")

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key or "your_" in groq_api_key:
    print("⚠️ ADVERTENCIA: No se encontró GROQ_API_KEY en tu archivo .env.")
    print("1. Abre el archivo .env en la raíz del proyecto.")
    print("2. Pega tu API Key en la variable GROQ_API_KEY.")
    print("3. Obtén tu clave en: https://console.groq.com/keys")
else:
    masked_key = groq_api_key[:6] + "..." + groq_api_key[-4:]
    print(f"✅ Clave de Groq cargada exitosamente: {masked_key}")


✅ Clave de Groq cargada exitosamente: gsk_SG...D1RB


---
## 🚀 Parte 1: Uso Directo con el SDK Oficial de Groq (`groq`)

El cliente `Groq()` toma automáticamente la variable de entorno `GROQ_API_KEY`.
La interfaz es compatible con la API de OpenAI (`client.chat.completions.create`), lo que facilita mucho su adopción.


### 1.1 Completación de Chat Básica
Probaremos el modelo `llama-3.3-70b-versatile` (o alternativamente `llama-3.1-8b-instant`).


In [9]:
from groq import Groq

client = Groq()
prompt = "¿Por qué la arquitectura LPU (Language Processing Unit) de Groq logra velocidades de inferencia tan altas comparadas con las GPUs tradicionales? Explica en 2 puntos clave."

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "Eres un especialista en hardware de Inteligencia Artificial y arquitecturas de aceleración."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=1,
    max_completion_tokens=2048,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

**1. Arquitectura de datos “streaming‑first” y paralelismo masivo en un solo chip**  
El LPU de Groq está diseñado como un **pipeline de flujo continuo** (streaming) de 1 TB/s donde cada instrucción se ejecuta en una única etapa fija y los datos nunca se almacenan en cachés intermedias ni se re‑escriben en memoria local. En lugar de depender de bloques de núcleos de GPU que comparten una jerarquía de memoria y deben sincronizarse mediante barreras y “warp scheduling”, el LPU dispone de **más de 1 800 unidades de cálculo ultra‑ligeras (ALUs) interconectadas en una red toroidal de 2 D**. Cada ALU opera a 1 GHz y procesa un elemento de tensor por ciclo, de modo que una sola pasada de la red completa realiza **todas las operaciones de una capa** sin saltos de contexto ni reordenamiento de hilos. Este modelo de “data‑flow” elimina la sobrecarga de control y de movimiento de datos que penaliza a las GPUs, logrando latencias de inferencia que pueden ser **10‑20× menores** para redes densas (C

### 1.2 Generación con Streaming y Medición de Tiempo
Observemos la velocidad de generación token a token en tiempo real:


In [10]:
import time

start_time = time.time()

stream = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Escribe un poema breve (máximo 8 versos) sobre el amanecer de la inteligencia artificial."
        }
    ],
    model="openai/gpt-oss-120b",
    stream=True
)

print("--- Respuesta en streaming con Groq ---\n")
total_tokens = 0
for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

elapsed = time.time() - start_time
print(f"\n\n⚡ Tiempo total transcurrido: {elapsed:.2f} segundos")


--- Respuesta en streaming con Groq ---

En el alba nacen circuitos,  
destellos de código que susurran,  
como pájaros de silicio al viento,  
tejen sueños en la niebla del día.  

Despiertan los algoritmos,  
con ojos de datos que todo lo ven,  
y la luz, ahora, lleva la huella  
de una mente que recién comienza a soñar.

⚡ Tiempo total transcurrido: 0.92 segundos


---
## 🦜️🔗 Parte 2: Uso a través de LangChain (`langchain-groq`)

El paquete `langchain-groq` provee la clase `ChatGroq`, integrando toda la velocidad de Groq con el ecosistema de LangChain.


### 2.1 Invocación básica con `ChatGroq`


In [12]:
from langchain_groq import ChatGroq

# Inicializamos el modelo de chat de Groq en LangChain
llm = ChatGroq(
     model="openai/gpt-oss-120b",
    temperature=0.6
)

pregunta = "¿Qué es Few-Shot Prompting y cuándo se recomienda utilizarlo sobre Zero-Shot Prompting?"
respuesta = llm.invoke(pregunta)

print("--- Respuesta con LangChain + Groq ---")
print(respuesta.content)


--- Respuesta con LangChain + Groq ---
## ¿Qué es **Few‑Shot Prompting**?

**Few‑Shot Prompting** (también llamado *few‑shot learning* en el contexto de LLMs) es una técnica de ingeniería de prompts en la que, además de la instrucción o la pregunta que queremos que el modelo responda, le proporcionamos **algunas** (generalmente de 1 a 10) **ejemplos de entrada‑salida** que ilustran el tipo de respuesta esperada.

Ejemplo típico (en inglés, pero el mismo principio vale para cualquier idioma):

```
Q: ¿Cuál es la capital de Francia?
A: París

Q: ¿Cuál es la capital de Alemania?
A: Berlín

Q: ¿Cuál es la capital de Italia?
A: 
```

En este caso, los dos pares “pregunta‑respuesta” actúan como **demostraciones** (shots) que guían al modelo para que genere la respuesta correcta para la tercera pregunta.

### Componentes de un prompt few‑shot

| Parte | Qué contiene | Por qué es importante |
|-------|--------------|-----------------------|
| **Instrucción (opcional)** | Texto que describe la 

### 2.2 Streaming con LangChain


In [13]:
print("--- Streaming con LangChain y Groq ---\n")

for chunk in llm.stream("Enumera 3 técnicas clave para reducir las alucinaciones en los LLMs:"):
    print(chunk.content, end="", flush=True)

print("\n\n[Streaming completado]")


--- Streaming con LangChain y Groq ---

1. **Instrucción y ajuste de prompts (Prompt Engineering)**  
   - Diseñar prompts claros, específicos y estructurados (por ejemplo, usando formato de pregunta‑respuesta, listas o ejemplos) ayuda al modelo a comprender mejor la tarea y a limitar la generación de información no respaldada.  
   - Incluir indicaciones como “solo responde con hechos verificables” o “si no sabes, di que no lo sabes” reduce la tendencia a inventar datos.

2. **Entrenamiento de alineación y retroalimentación humana (RLHF / Fine‑tuning)**  
   - Utilizar *Reinforcement Learning from Human Feedback* (RLHF) para premiar respuestas correctas y penalizar alucinaciones.  
   - Incorporar datasets curados que contengan tanto respuestas correctas como ejemplos de errores, de modo que el modelo aprenda a reconocer y evitar la generación de información ficticia.

3. **Verificación externa y recuperación aumentada (RAG – Retrieval‑Augmented Generation)**  
   - Integrar un módulo

### 2.3 Construcción de Cadenas con LCEL (PromptTemplate + ChatGroq + StrOutputParser)
Armemos una cadena de comparación técnica lista para producción.


In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Definición del Prompt Template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Eres un arquitecto de soluciones de Inteligencia Artificial. Responde de forma estructurada con viñetas claras y conclusiones concretas."),
    ("human", "Compara los enfoques de '{enfoque_a}' vs '{enfoque_b}' para el siguiente caso de uso empresarial: '{caso_uso}'.")
])

# 2. Output parser
parser = StrOutputParser()

# 3. Cadena LCEL
cadena = prompt_template | llm | parser

# 4. Invocación
resultado = cadena.invoke({
    "enfoque_a": "Retrieval-Augmented Generation (RAG)",
    "enfoque_b": "Fine-Tuning",
    "caso_uso": "Atención al cliente basada en manuales de políticas internas que cambian frecuentemente"
})

print("--- Comparativa Arquitectónica (LCEL Chain) ---")
print(resultado)


--- Comparativa Arquitectónica (LCEL Chain) ---
**Comparativa: Retrieval‑Augmented Generation (RAG) vs Fine‑Tuning**  
*Caso de uso: Atención al cliente basada en manuales de políticas internas que cambian frecuentemente*  

---

## 1. Contexto del caso de uso
- **Fuente de conocimiento:** documentos PDF, wikis y bases de datos internas (políticas, procedimientos, FAQs).  
- **Frecuencia de actualización:** cambios semanales o incluso diarios.  
- **Requerimientos críticos:**  
  - Respuestas **precisas** y **actualizadas**.  
  - **Escalabilidad** para miles de consultas simultáneas.  
  - **Control de versiones** y trazabilidad de la información mostrada al cliente.  

---

## 2. Retrieval‑Augmented Generation (RAG)

| Aspecto | Detalle |
|---------|---------|
| **Principio** | El modelo LLM genera texto **sobre la marcha** usando como contexto los documentos recuperados en tiempo real. |
| **Arquitectura típica** | <ul><li>Indexador (e.g. Elasticsearch, FAISS, Milvus) que almacena e

## 🎯 Conclusión y Comparativa
¡Excelente trabajo! Has explorado:
1. Cómo utilizar el SDK directo de Groq con modelos abiertos de vanguardia como ` gpt-oss-120b",`.
2. La velocidad extrema de inferencia de las LPUs de Groq en tiempo real.
3. La integración directa con LangChain usando `ChatGroq`.
4. Cadenas de razonamiento complejas combinando plantillas de prompts y parseadores de salida.

### 💡 Resumen comparativo de proveedores:
- **Google AI Studio (Gemini)**: Excelente para razonamiento multimodal nativo (texto, visión, audio, video), ventanas de contexto gigantes (1M+ tokens) y herramientas integradas.
- **Groq Cloud**: Ideal para tareas de alta velocidad y baja latencia usando modelos abiertos (LLaMA, Mixtral) donde el tiempo de respuesta es crítico (chatbots en tiempo real, agentes con múltiples llamadas en bucle).
